# Interface Gradio — Narrative Master

Interface de jeu pour le Game Master. Par défaut, elle utilise le modèle de base ;
pour utiliser le modèle adapté, indiquer le dossier de l'adaptateur LoRA dans `ADAPTER_PATH`.

Exécuter les cellules dans l'ordre ; la dernière lance l'interface dans le notebook.

**Sur Google Colab** : choisir un environnement d'exécution avec GPU
(*Exécution → Modifier le type d'exécution → GPU T4*).

## 0. Installation

In [1]:
import os
import sys

# Sur Colab : installer gradio et peft, puis récupérer le dépôt (il contient l'adaptateur LoRA)
if "google.colab" in sys.modules:
    !pip uninstall -q -y torchao  # version de Colab incompatible avec peft
    !pip install -q gradio peft
    if not os.path.exists("/content/NLP_narrative-game"):
        !git clone -q --depth 1 https://github.com/2ec3/NLP_narrative-game.git /content/NLP_narrative-game
    %cd /content/NLP_narrative-game/notebooks

/content/NLP_narrative-game/notebooks


## 1. Chargement du modèle

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Chemin vers votre meilleur modèle (r=8, lr=2e-4) depuis le dossier notebooks/
ADAPTER_PATH = "../model_adapter/adapter_r8_lr2e-4"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

if ADAPTER_PATH is not None:
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)

print("Appareil :", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Appareil : cuda:0


## 2. Paramètres

Le prompt système doit être **identique** à celui des données d'entraînement.

In [3]:
SYSTEM_PROMPT = (
    "Tu es le Game Master d’un jeu narratif dans un univers fantastique avec des humains, "
    "des elfes, des gobelins et de la magie. Tu proposes à chaque étape 2 à 3 actions possibles. "
    "Tu dois respecter les événements précédents et l’état du monde. Tu refuses les actions "
    "impossibles ou incohérentes. Le joueur ne peut pas décider à la place des PNJ ou des ennemis, "
    "ni modifier directement les éléments du décor. Si une action est ambiguë, tu demandes au "
    "joueur de préciser son intention."
)

MAX_NEW_TOKENS = 300
TEMPERATURE = 1.0
TOP_K = 30
TOP_P = 0.8

MAX_TURNS = 10     # nombre maximal d'actions du joueur dans une partie
HISTORY_SIZE = 12  # nombre de messages envoyés au modèle (6 échanges GM + joueur)

# Ajouté au dernier message du joueur pour que le modèle termine l'histoire
END_HINT = "\n\n(Dernier tour : conclus l'aventure en quelques phrases, sans proposer de nouvelles actions.)"

## 3. Génération d'une réponse

In [4]:
def cut_last_sentence(text):
    """Garde seulement les phrases complètes (si la réponse a été coupée par max_new_tokens)."""
    last = -1
    for c in [".", "!", "?", "\n"]:
        last = max(last, text.rfind(c))
    if last == -1:
        return text
    return text[:last + 1].strip()


def generate_reply(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        top_p=TOP_P
    )

    # on garde seulement les nouveaux tokens (sans le prompt)
    new_ids = generated_ids[0][len(inputs.input_ids[0]):]
    reply = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    if len(new_ids) >= MAX_NEW_TOKENS:
        reply = cut_last_sentence(reply)
    return reply


def build_messages(history):
    """Prompt système + derniers messages seulement (historique de taille contrôlée)."""
    return [{"role": "system", "content": SYSTEM_PROMPT}] + history[-HISTORY_SIZE:]


def count_turns(history):
    return len([m for m in history if m["role"] == "user"])

## 4. Interface

In [5]:
import gradio as gr

DESCRIPTION = f"""
# Narrative Master
Je suis le **Game Master** d'une aventure de fantasy (humains, elfes, gobelins, magie).
À chaque tour, je décris la scène et propose **2 ou 3 actions (A, B, C)** : répondez par une lettre
ou décrivez votre choix avec vos mots.

**Limites** : une partie dure au maximum **{MAX_TURNS} tours**. Les actions impossibles ou hors
de l'univers sont refusées : vous ne pouvez pas faire apparaître de l'or, contrôler les PNJ ou
modifier le décor. Seuls les derniers échanges sont transmis au modèle.
"""

GAME_OVER = "*La partie est terminée. Cliquez sur « Nouvelle partie » pour recommencer.*"


def start_game():
    # On envoie un message déclencheur caché pour lancer l'aventure
    opening = generate_reply(build_messages([{"role": "user", "content": "Commence l'aventure."}]))
    history = [{"role": "assistant", "content": opening}]
    return history, history, f"Tour 0 / {MAX_TURNS}"


def respond(message, history):
    turn = count_turns(history)

    # message vide ou partie terminée : on ne fait rien
    if message.strip() == "" or turn >= MAX_TURNS:
        return "", history, history, f"Tour {turn} / {MAX_TURNS}"

    history = history + [{"role": "user", "content": message}]
    turn = turn + 1

    messages = build_messages(history)
    if turn == MAX_TURNS:
        messages[-1] = {"role": "user", "content": message + END_HINT}

    reply = generate_reply(messages)
    history = history + [{"role": "assistant", "content": reply}]

    if turn == MAX_TURNS:
        history = history + [{"role": "assistant", "content": GAME_OVER}]
    return "", history, history, f"Tour {turn} / {MAX_TURNS}"


with gr.Blocks(title="Narrative Master") as demo:
    history = gr.State([])  # historique de la partie

    gr.Markdown(DESCRIPTION)
    turn_info = gr.Markdown(f"Tour 0 / {MAX_TURNS}")
    chatbot = gr.Chatbot(label="Partie", height=480)
    msg = gr.Textbox(placeholder="Votre action (ex. « A » ou « J'entre dans le château »)", show_label=False)
    send = gr.Button("Envoyer", variant="primary")
    reset = gr.Button("Nouvelle partie")

    msg.submit(respond, [msg, history], [msg, history, chatbot, turn_info])
    send.click(respond, [msg, history], [msg, history, chatbot, turn_info])
    reset.click(start_game, None, [history, chatbot, turn_info])
    chatbot.clear(start_game, None, [history, chatbot, turn_info])  # la corbeille du chat = nouvelle partie
    demo.load(start_game, None, [history, chatbot, turn_info])      # nouvelle partie à l'ouverture

demo.launch(share=False)  # pas de lien public

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>